In [1]:
import os
from typing import Annotated, List, TypedDict, Optional, Union
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\ASUS\OneDrive\Desktop\DeepForge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# The specialized framework for deep reasoning agents
from deepagents import create_deep_agent

In [3]:
load_dotenv()

True

In [4]:
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    paper_path: Optional[str]
    scientific_logic: Optional[str]
    implementation_code: Optional[str]

print("Environment and State initialized.")

Environment and State initialized.


In [5]:
llm= ChatGoogleGenerativeAI(
    model= "gemini-2.0-flash",
    temperature= 0
)

In [6]:
research_agent= create_deep_agent(
    model= llm,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your job is to read research papers, extract the mathematical core, "
        "and explain the architecture in plain pseudocode. "
        "Use your planning tools to break down the paper analysis step-by-step."
    )
)
print("Deep Research Agent created.")

Deep Research Agent created.


In [7]:
import arxiv
import fitz  
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool

In [8]:
# 1. arXiv Search and Download Tool (Fixed for Windows & Modern API)
@tool
def download_research_paper(query: str) -> str:
    """
    Searches arXiv for a paper, downloads the PDF to the '../data/' folder, 
    and returns the local path to the file.
    """
    download_path = "../data/"
    os.makedirs(download_path, exist_ok=True)

    # Use the modern Client for stability
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=1, sort_by=arxiv.SortCriterion.Relevance)
    
    try:
        results = list(client.results(search))
        if not results: return "No papers found."
        
        paper = results[0]
        # FIX: Remove ALL illegal Windows characters ( : / \ * " < > | )
        clean_title = "".join([c if c.isalnum() or c==' ' else '' for c in paper.title])
        filename = f"{clean_title.replace(' ', '_')}.pdf"
        
        full_path = paper.download_pdf(dirpath=download_path, filename=filename)
        return f"Paper downloaded to: {full_path}"
    except Exception as e:
        return f"Error during download: {str(e)}"

print("arXiv tool is now Windows-safe and Modern!")

arXiv tool is now Windows-safe and Modern!


In [9]:
# 2. PDF Reading Tool
@tool
def read_pdf_content(file_path: str) -> str:
    """
    Opens a PDF file and extracts the text content. 
    Useful for reading downloaded research papers.
    """
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [10]:
# 3. Web Search Tool (Tavily)
web_search = TavilySearchResults(k=3)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21996\3970997022.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(k=3)


In [11]:
# Combine all tools
tools = [download_research_paper, read_pdf_content, web_search]
print("ArXiv, PDF, and Tavily tools defined.")

ArXiv, PDF, and Tavily tools defined.


In [12]:
# 1. Re-initialize the Researcher Agent with tools
# We give it our 'tools' list so it can search, download, and read.
researcher_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt= (
        "You are the Paper-to-Production Analyzer. "
        "Your mission is to find research papers and extract their implementation core. "
        "Step 1: Use 'download_research_paper' to get the PDF. "
        "Step 2: Use 'read_pdf_content' to understand the math and logic. "
        "Step 3: Output a structured explanation of the algorithm, variables, and pseudocode."
    )
)
print("Research Agent re-Linked with tools.")

Research Agent re-Linked with tools.


In [13]:
# trigger a "Deep" research session
inputs = {
    "messages": [
        ("user", "Research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.")
    ]
}

config = {"configurable": {"thread_id": "research-task-1"}}
print("Starting Deep Research. Watch the agent plan and execute...")

for chunk in research_agent.stream(inputs, config):
    print(chunk)

Starting Deep Research. Watch the agent plan and execute...
{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.', additional_kwargs={}, response_metadata={}, id='2c577d7d-4d59-4396-93c0-b7a592a3b893')])}}
{'model': {'messages': [AIMessage(content='Okay, I will research the QLoRA paper and explain how the 4-bit NormalFloat quantization works.', additional_kwargs={'function_call': {'name': 'write_todos', 'arguments': '{"todos": [{"content": "Research the QLoRA paper to understand the 4-bit NormalFloat quantization method.", "status": "in_progress"}, {"status": "pending", "content": "Explain the 4-bit NormalFloat quantization method based on the research."}]}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cfbe5-8c72-7013-bb8b-3cca04448b7e-0', tool_calls=[{'name

In [14]:
# This agent takes the research findings and plans the Code structure.
designer_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt= (
        "You are the Architecture Designer. "
        "Your goal is to take research findings and create a high-level Design Spec. "
        "1. Define the Python classes and methods needed. "
        "2. Specify the exact mathematical formulas for each method. "
        "3. Decide on the libraries to use (e.g., torch, numpy). "
        "Output ONLY the design specification, not the actual code yet."
    )
)
print("Architecture Designer Agent created.")

Architecture Designer Agent created.


In [15]:
workflow = StateGraph(ResearchState)

# 1. Add Nodes
# IMPORTANT: We use .invoke()["messages"] to return a dict that matches our State
workflow.add_node("researcher", lambda state: {"messages": researcher_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("designer", lambda state: {"messages": designer_agent.invoke({"messages": state["messages"]})["messages"]})

# 2. Define the Flow (Initial)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "designer")
workflow.add_edge("designer", END)

# 3. Compile
app = workflow.compile()
print("Node logic fixed for Researcher & Designer.")

Node logic fixed for Researcher & Designer.


In [16]:
# The Coder Agent
# Its only job is to turn the Design Spec into high-quality, executable Python code.
coder_agent= create_deep_agent(
    model= llm,
    tools= tools,
    system_prompt=(
        "You are the Lead Programmer for DeepForge. "
        "Your goal is to implement the Architecture Designer's specification. "
        "1. Write complete, well-documented Python code. "
        "2. Include a 'main' block or test case to verify the implementation. "
        "3. Ensure you follow research paper naming conventions for variables. "
        "Output ONLY the Python code inside a markdown block."
    )
)
print("Coder Agent created.")

Coder Agent created.


In [17]:
# 1. Add the Coder node to the existing workflow
workflow.add_node("coder", lambda state: {"messages": coder_agent.invoke({"messages": state["messages"]})["messages"]})

# 2. Re-route the Designer to the Coder instead of END
# We use a new edge to extend the chain
workflow.add_edge("designer", "coder")
workflow.add_edge("coder", END)

# 3. Re-compile the Full Pipeline
deep_agent_pipeline = workflow.compile()
print("✅ Pipeline evolved! Coder added successfully.")

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


✅ Pipeline evolved! Coder added successfully.


In [18]:
final_inputs= {
    "messages": [
        ("user", "Research the 'LoRA: Low-Rank Adaptation' paper and implement a simple LoRA linear layer in PyTorch.")
    ]
}
print("Executing DEEP PIPELINE: Reasearching-> Designing-> Coding...")

# Run the pipeline
for event in deep_agent_pipeline.stream(final_inputs, config):
    # This will show you exactly which agent is working
    for node_name, output in event.items():
        print(f"\n--- [AGENT: {node_name.upper()}] ---")
        # We print the last message content from that agent
        print(output["messages"][-1].content)

Executing DEEP PIPELINE: Reasearching-> Designing-> Coding...

--- [AGENT: RESEARCHER] ---
Here's a breakdown of the LoRA implementation:

**Algorithm Explanation:**

LoRA approximates weight updates by decomposing them into low-rank matrices. Instead of directly training the large weight matrix W0, LoRA introduces two smaller matrices, A and B, such that the update ΔW ≈ BA. During training, W0 is frozen, and only A and B are trained. This significantly reduces the number of trainable parameters.

**Variables:**

*   `in_features`: The number of input features.
*   `out_features`: The number of output features.
*   `rank` (r): The rank of the low-rank matrices A and B. This determines the size of the trainable parameters.
*   `lora_alpha`: A scaling factor used to adjust the magnitude of the LoRA update.
*   `W0`: The original pre-trained weight matrix (frozen during LoRA training).
*   `A`: The first low-rank matrix (trainable).
*   `B`: The second low-rank matrix (trainable).

**Pseu

In [19]:
from langchain_experimental.utilities import PythonREPL
from langchain_core.tools import tool 

# Initialize the Python Runtime
repl = PythonREPL()

@tool
def execute_python_code(code: str) -> str:
    """Executes Python code and returns output. Detects errors inside the output."""
    try:
        clean_code = code.replace("```python", "").replace("```", "")
        # Force numpy import in the execution context if missing
        if "import numpy" not in clean_code and "np." in clean_code:
            clean_code = "import numpy as np\n" + clean_code
            
        result = repl.run(clean_code)
        
        # Check if result contains error keywords (Traceback, NameError, etc.)
        if "Error" in result or "Traceback" in result:
            return f"Execution Failed:\n{result}"
            
        return f"Execution Success:\n{result if result else 'Code ran with no output (Success).'}"
    except Exception as e:
        return f"Execution Failed:\n{str(e)}"

# Add the new tool to our collection
if execute_python_code not in tools:
    tools.append(execute_python_code)
print("Execution Tool fixed and linked safely.")

Execution Tool fixed and linked safely.


In [20]:
# Master Pipeline Setup - Run this to rebuild the whole graph
workflow = StateGraph(ResearchState)

# 1. Define all nodes
workflow.add_node("researcher", lambda state: {"messages": researcher_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("designer", lambda state: {"messages": designer_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("coder", lambda state: {"messages": coder_agent.invoke({"messages": state["messages"]})["messages"]})

# Executor node logic
# Executor node logic (Fixed to search backwards)
def executor_node(state: ResearchState):
    # Scan history for the most recent python code block
    code = None
    for msg in reversed(state["messages"]):
        if "```python" in msg.content:
            code = msg.content.split("```python")[1].split("```")[0]
            break
            
    if code:
        result = execute_python_code.invoke(code)
        return {"messages": [HumanMessage(content=f"EXECUTOR RESULTS:\n{result}")]}
        
    return {"messages": [HumanMessage(content="ERROR: No code block found in conversation history.")]}

workflow.add_node("executor", executor_node)

# 2. Define the Flow
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "designer")
workflow.add_edge("designer", "coder")
workflow.add_edge("coder", "executor")

# Conditional logic for self-healing
def should_continue(state: ResearchState):
    last_message = state["messages"][-1].content
    if "Execution Success" in last_message:
        return END
    return "coder"

workflow.add_conditional_edges("executor", should_continue)

# 3. Compile
deep_agent_pipeline = workflow.compile()
print("Master Pipeline compiled successfully!")

Master Pipeline compiled successfully!


In [21]:
# The Reviewer Agent
reviewer_agent= create_deep_agent(
    model= llm,
    system_prompt= (
        "You are the Peer Reviewer. "
        "Your job is to audit the generated code against the original research logic. "
        "1. Check if the mathematical formulas match the paper. "
        "2. Look for missing initialization steps (e.g., zero-init for weights). "
        "3. Verify if variable names match the paper's notation. "
        "Output a 'Review Report'. If the code is perfect, say 'LGTM' (Looks Good To Me). "
        "If there are errors, explain them clearly so the Coder can fix them."
    )
)
print("Reviewer Agent created.")

Reviewer Agent created.


In [22]:
# Master Pipeline Setup v2 - The "Deep Reasoning" version
workflow= StateGraph(ResearchState)

# 1. Nodes
workflow.add_node("researcher", lambda state: {"messages": researcher_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("designer", lambda state: {"messages": designer_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("coder", lambda state: {"messages": coder_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("reviewer", lambda state: {"messages": reviewer_agent.invoke({"messages": state["messages"]})["messages"]})

# Executor node logic (Fixed to search backwards)
def executor_node(state: ResearchState):
    # Scan history for the most recent python code block
    code = None
    for msg in reversed(state["messages"]):
        if "```python" in msg.content:
            code = msg.content.split("```python")[1].split("```")[0]
            break
            
    if code:
        result = execute_python_code.invoke(code)
        return {"messages": [HumanMessage(content=f"EXECUTOR RESULTS:\n{result}")]}
        
    return {"messages": [HumanMessage(content="ERROR: No code block found in conversation history.")]}

workflow.add_node("executor", executor_node)

# 2. Advanced Flow Logic
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "designer")
workflow.add_edge("designer", "coder")
workflow.add_edge("coder", "reviewer")

# Loop logic for Reviewer
def verify_logic(state: ResearchState):
    last_review= state["messages"][-1].content
    if "LGTM" in last_review or "Looks Good To Me" in last_review:
        return "executor" # If logic is good, go to execution
    return "coder" # If logic is bad, go back to Coder to fix

workflow.add_conditional_edges("reviewer", verify_logic)

# Loop logic for Executor (Success check)
def verify_execution(state: ResearchState):
    last_exec = state["messages"][-1].content
    # FIX: Changed "Sucess" -> "Success"
    if "Execution Success" in last_exec:
        return END
    return "coder"

workflow.add_conditional_edges("executor", verify_execution)

# 3. Final Compile
deep_agent_pipeline = workflow.compile()
print("Professional Research -> Design -> Code -> Review -> Execute Pipeline is LIVE!")

Professional Research -> Design -> Code -> Review -> Execute Pipeline is LIVE!


In [23]:
@tool
def save_code_to_production(code: str, filename: str) -> str:
    """
    Saves the finalized, verified code to a file in the 'src/' directory.
    Use this only after the code has passed Review and Execution tests.
    """
    try:
        # Ensure the directory exists
        os.makedirs("../src", exist_ok=True)
        
        # Clean the code and filename
        clean_code = code.replace("```python", "").replace("```", "")
        file_path = os.path.join("../src", filename)
        
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(clean_code)
            
        return f"Production-ready code saved to: {file_path}"
    except Exception as e:
        return f"Failed to save code: {str(e)}"

# Add to our tool library
if save_code_to_production not in tools:
    tools.append(save_code_to_production)
print("Production Persistence tool added safely.")

Production Persistence tool added safely.


In [24]:
# The Benchmarker Agent 
benchmarker_agent= create_deep_agent(
    model= llm,
    tools= tools, 
    system_prompt= (
        "You are the Lead Experiment Scientist. "
        "Your goal is to verify the 'Efficiency' claims of the research paper. "
        "1. Write a script that imports the saved module. "
        "2. Calculate metrics: Number of trainable parameters, Memory footprint, or Throughput. "
        "3. Output an 'Experiment Report' summarizing the efficiency gains."
    )
)
print("Benchmarker Agent created.")

Benchmarker Agent created.


In [25]:
# Master Pipeline Setup v3 - FINAL PRODUCTION FORM
workflow = StateGraph(ResearchState)

# 1. All Nodes
workflow.add_node("researcher", lambda state: {"messages": researcher_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("designer", lambda state: {"messages": designer_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("coder", lambda state: {"messages": coder_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("reviewer", lambda state: {"messages": reviewer_agent.invoke({"messages": state["messages"]})["messages"]})
workflow.add_node("executor", executor_node) 
workflow.add_node("benchmarker", lambda state: {"messages": benchmarker_agent.invoke({"messages": state["messages"]})["messages"]})

# 2. Complete Flow Logic
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "designer")
workflow.add_edge("designer", "coder")
workflow.add_edge("coder", "reviewer")

# Reviewer Logic
def verify_review(state):
    lc = state["messages"][-1].content
    return "executor" if ("LGTM" in lc or "Looks Good" in lc) else "coder"

workflow.add_conditional_edges("reviewer", verify_review)

# Executor Logic (New: If success, save it and benchmark!)
def verify_exec(state):
    lc = state["messages"][-1].content
    return "benchmarker" if "Execution Success" in lc else "coder"

workflow.add_conditional_edges("executor", verify_exec)

# Benchmarker finishes the task
workflow.add_edge("benchmarker", END)

# 3. Final Compile
deep_agent_pipeline = workflow.compile()
print("MASTER PIPELINE ASSEMBLED. Flow: Research -> Design -> Code -> Review -> Test -> Benchmark -> Done.")

MASTER PIPELINE ASSEMBLED. Flow: Research -> Design -> Code -> Review -> Test -> Benchmark -> Done.


In [26]:
# ==========================================
# THE MASTER-SYNC PRODUCTION PIPELINE
# ==========================================

from langchain_core.messages import AIMessage, SystemMessage

print("Building Master Pipeline (Stable & Fast)...")

# 1. SPECIALIZED AGENTS & TOOLS
coding_tools = [execute_python_code, save_code_to_production]

coder_agent = create_deep_agent(llm, tools=coding_tools, 
    system_prompt="Coder: Implement the NF4 class. IMPORTANT: You MUST include a 'if __name__ == \"__main__\":' block that prints the MSE result.")

# 2. DEFINING ALL NODE FUNCTIONS (THE BRAINS)
def run_researcher(state):
    print("\n[STEP 1] RESEARCHER starting...")
    res = researcher_agent.invoke({"messages": state["messages"]})
    msg = res["messages"][-1]
    print(f"RESEARCHER DONE.")
    return {"messages": [msg]}

def run_designer(state):
    print("[STEP 2] DESIGNER starting...")
    prompt = [SystemMessage(content="Design the 'NormalFloat4Converter' class structure (quantize/dequantize).")] + state["messages"]
    res = llm.invoke(prompt)
    print(f"DESIGNER DONE.")
    return {"messages": [res]}

def run_coder(state):
    print("[STEP 3] CODER starting...")
    res = coder_agent.invoke({"messages": state["messages"]})
    msg = res["messages"][-1]
    print(f"CODER DONE.")
    return {"messages": [msg]}

def run_reviewer(state):
    print("[STEP 4] REVIEWER starting...")
    prompt = [SystemMessage(content="Audit the code. If perfect, say 'LGTM'. If not, explain errors.")] + state["messages"]
    res = llm.invoke(prompt)
    print(f"REVIEWER DONE.")
    return {"messages": [res]}

def run_benchmarker(state):
    print("[STEP 5] BENCHMARKER starting...")
    prompt = [SystemMessage(content="Summarize the final MSE results and confirm production-readiness.")] + state["messages"]
    res = llm.invoke(prompt)
    print("TASK FINISHED.")
    return {"messages": [res]}

# 3. ROUTING LOGIC (THE NERVOUS SYSTEM)
def verify_review_final(state):
    lc = state["messages"][-1].content.upper()
    return "executor" if ("LGTM" in lc or "LOOKS GOOD" in lc) else "coder"

def verify_execution_final(state):
    last_msg = state["messages"][-1].content.upper()
    # Must find SUCCESS and NO error keywords
    if "SUCCESS" in last_msg and "FAILED" not in last_msg:
        return "benchmarker"
    return "coder"

# 4. GRAPH ASSEMBLY (THE BODY)
builder = StateGraph(ResearchState)
builder.add_node("researcher", run_researcher)
builder.add_node("designer", run_designer)
builder.add_node("coder", run_coder)
builder.add_node("reviewer", run_reviewer)
builder.add_node("executor", executor_node) 
builder.add_node("benchmarker", run_benchmarker)

builder.set_entry_point("researcher")
builder.add_edge("researcher", "designer")
builder.add_edge("designer", "coder")
builder.add_edge("coder", "reviewer")
builder.add_conditional_edges("reviewer", verify_review_final)
builder.add_conditional_edges("executor", verify_execution_final)
builder.add_edge("benchmarker", END)

deep_agent_pipeline = builder.compile()
print("MASTER PIPELINE READY: No errors, maximum speed.")

Building Master Pipeline (Stable & Fast)...
MASTER PIPELINE READY: No errors, maximum speed.


In [27]:
# ==========================================
# THE STABLE GRAND TEST
# ==========================================

final_inputs = {
    "messages": [
        ("user", "IMPLEMENT NF4: Use these 16 centroids: [-1.0, -0.6961, -0.5250, -0.3949, -0.2844, -0.1847, -0.0910, 0.0, 0.0795, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7229, 1.0]. Implement the NormalFloat4Converter class and test MSE.")
    ]
}

config = {"configurable": {"thread_id": "STABLE-PRODUCTION-v1"}}
print("INITIATING STABLE TEST: Watch the clean node-by-node flow...\n")

for event in deep_agent_pipeline.stream(final_inputs, config):
    for node_name, output in event.items():
        print(f"\n{'='*20} [{node_name.upper()} COMPLETED] {'='*20}")
        if "messages" in output:
            print(output["messages"][-1].content)

INITIATING STABLE TEST: Watch the clean node-by-node flow...


[STEP 1] RESEARCHER starting...
RESEARCHER DONE.

==================== [RESEARCHER COMPLETED] ====================
The NormalFloat4Converter class has been implemented and tested for MSE. The code has been written to a file named `/NormalFloat4Converter.py`.
```python
import numpy as np

class NormalFloat4Converter:
    def __init__(self):
        self.centroids = np.array([-1.0, -0.6961, -0.5250, -0.3949, -0.2844, -0.1847, -0.0910, 0.0, 0.0795, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7229, 1.0], dtype=np.float32)

    def quantize(self, data):
        # Flatten the input data
        flattened_data = data.flatten()

        # Assign each value to the nearest centroid
        quantized_data = np.zeros_like(flattened_data, dtype=np.uint8)
        for i, value in enumerate(flattened_data):
            distances = np.abs(self.centroids - value)
            quantized_data[i] = np.argmin(distances)

        # Reshape to origin

Python REPL can execute arbitrary code. Use with caution.


REVIEWER DONE.

==================== [REVIEWER COMPLETED] ====================


LGTM

==================== [EXECUTOR COMPLETED] ====================
EXECUTOR RESULTS:
Execution Success:
Code ran with no output (Success).
[STEP 5] BENCHMARKER starting...
TASK FINISHED.

==================== [BENCHMARKER COMPLETED] ====================
Okay, the execution was successful with no output. This likely means the script ran without errors, but the test case within the `if __name__ == '__main__':` block didn't produce any printed output because it was suppressed or commented out. To confirm the MSE and production-readiness, we need to modify the script to print the MSE value explicitly and run it again.

Here's the modified script:

```python
import numpy as np

class NormalFloat4Converter:
    def __init__(self):
        self.centroids = np.array([-1.0, -0.6961, -0.5250, -0.3949, -0.2844, -0.1847, -0.0910, 0.0, 0.0795, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7229, 1.0], dtype=np.float32)

 